# Spark Spanner Connector — Write Demo (Dataproc)

This notebook demonstrates the connector's write capabilities on Dataproc:

- **Part 1** — DataFrame API writes (append, overwrite, mutation types)
- **Part 2** — Spark Catalog (CREATE TABLE, INSERT INTO, SELECT, DROP TABLE)
- **Part 3** — Struct-to-JSON mapping (Spark structs ↔ Spanner JSON columns)

### Prerequisites
- Dataproc cluster with the Spark Spanner Connector JAR installed.
- GCP credentials available (default on Dataproc).

## Configuration

In [ ]:
from pyspark.sql.types import StructType, StructField, LongType, StringType, DoubleType

SPANNER_OPTS = {
    "projectId":  "spark-spanner-connector",
    "instanceId": "spark-spanner-testing",
    "databaseId": "repo-test",
}

# Register the Spanner catalog (works on Dataproc, not on Databricks Unity Catalog)
spark.conf.set("spark.sql.catalog.spanner",
               "com.google.cloud.spark.spanner.SpannerCatalog")
spark.conf.set("spark.sql.catalog.spanner.projectId",  SPANNER_OPTS["projectId"])
spark.conf.set("spark.sql.catalog.spanner.instanceId", SPANNER_OPTS["instanceId"])
spark.conf.set("spark.sql.catalog.spanner.databaseId", SPANNER_OPTS["databaseId"])

print(f"Spanner: {SPANNER_OPTS['projectId']}/{SPANNER_OPTS['instanceId']}/{SPANNER_OPTS['databaseId']}")
print("✓ Spanner catalog registered.")

---
# Part 1 — DataFrame API Writes

## 1a. Create the target table

In [ ]:
%%sql
DROP TABLE IF EXISTS spanner.demo_df_write

In [ ]:
%%sql
CREATE TABLE spanner.demo_df_write (
    id     BIGINT NOT NULL,
    name   STRING,
    dept   STRING,
    salary DOUBLE
) USING `cloud-spanner`
TBLPROPERTIES('primaryKeys' = 'id')

## 1b. Append mode

The default — inserts rows into the existing table.

In [ ]:
schema = StructType([
    StructField("id",     LongType(),   False),
    StructField("name",   StringType(), True),
    StructField("dept",   StringType(), True),
    StructField("salary", DoubleType(), True),
])

data = [
    (1, "Alice",   "Engineering", 120000.0),
    (2, "Bob",     "Marketing",    95000.0),
    (3, "Charlie", "Engineering", 130000.0),
]
df = spark.createDataFrame(data, schema)

(df.write.format("cloud-spanner")
    .options(**SPANNER_OPTS)
    .option("table", "demo_df_write")
    .mode("append")
    .save())

print("✓ Wrote 3 rows with append mode.")

In [ ]:
%%sql
SELECT * FROM spanner.demo_df_write ORDER BY id

## 1c. Mutation type: INSERT

The `mutationType` option controls the Spanner mutation used:
- `insert` — fails if the row already exists
- `update` — fails if the row does not exist
- `insert_or_update` (default) — upserts
- `replace` — replaces the entire row

In [ ]:
new_row = spark.createDataFrame([(4, "Diana", "Sales", 105000.0)], schema)

(new_row.write.format("cloud-spanner")
    .options(**SPANNER_OPTS)
    .option("table", "demo_df_write")
    .option("mutationType", "insert")
    .mode("append")
    .save())

print("✓ Inserted row id=4 with mutationType='insert'.")

In [ ]:
%%sql
SELECT * FROM spanner.demo_df_write ORDER BY id

## 1d. Overwrite mode (truncate)

`mode("overwrite")` with `overwriteMode=truncate` deletes all existing rows, then writes the new data. The table schema is preserved.

In [ ]:
%%sql
DROP TABLE IF EXISTS spanner.demo_overwrite

In [ ]:
%%sql
CREATE TABLE spanner.demo_overwrite (
    id   BIGINT NOT NULL,
    name STRING
) USING `cloud-spanner`
TBLPROPERTIES('primaryKeys' = 'id')

In [ ]:
small_schema = StructType([
    StructField("id",   LongType(),   False),
    StructField("name", StringType(), True),
])

initial = spark.createDataFrame([(1, "old-row-1"), (2, "old-row-2")], small_schema)
(initial.write.format("cloud-spanner")
    .options(**SPANNER_OPTS)
    .option("table", "demo_overwrite")
    .mode("append")
    .save())

print("✓ Seeded demo_overwrite with 2 rows.")

In [ ]:
%%sql
SELECT * FROM spanner.demo_overwrite ORDER BY id

In [ ]:
replacement = spark.createDataFrame([(10, "new-row-10"), (20, "new-row-20")], small_schema)

(replacement.write.format("cloud-spanner")
    .options(**SPANNER_OPTS)
    .option("table", "demo_overwrite")
    .option("overwriteMode", "truncate")
    .mode("overwrite")
    .save())

print("✓ Overwrote with 2 new rows (truncate mode).")

In [ ]:
%%sql
SELECT * FROM spanner.demo_overwrite ORDER BY id

---
# Part 2 — Spark Catalog (SQL)

## 2a. CREATE TABLE

In [ ]:
%%sql
DROP TABLE IF EXISTS spanner.demo_catalog

In [ ]:
%%sql
CREATE TABLE spanner.demo_catalog (
    id     BIGINT NOT NULL,
    city   STRING,
    temp_f DOUBLE
) USING `cloud-spanner`
TBLPROPERTIES('primaryKeys' = 'id')

## 2b. INSERT INTO

In [ ]:
%%sql
INSERT INTO spanner.demo_catalog VALUES
    (1, 'San Francisco', 62.0),
    (2, 'New York', 45.0),
    (3, 'Austin', 85.0)

## 2c. SELECT

In [ ]:
%%sql
SELECT * FROM spanner.demo_catalog ORDER BY id

## 2d. writeTo().create() — ErrorIfExists

In [ ]:
%%sql
DROP TABLE IF EXISTS spanner.demo_catalog_eie

In [ ]:
df = spark.createDataFrame(
    [(100, "Seattle", 55.0)],
    StructType([
        StructField("id",     LongType(),   False),
        StructField("city",   StringType(), True),
        StructField("temp_f", DoubleType(), True),
    ]),
)

df.writeTo("spanner.demo_catalog_eie").tableProperty("primaryKeys", "id").create()
print("✓ Created demo_catalog_eie and wrote 1 row.")

In [ ]:
%%sql
SELECT * FROM spanner.demo_catalog_eie ORDER BY id

In [ ]:
# Second create should fail (table already exists)
try:
    df.writeTo("spanner.demo_catalog_eie").tableProperty("primaryKeys", "id").create()
    print("✗ Expected error was NOT raised!")
except Exception as e:
    print(f"✓ Second create() correctly failed: {e}")

## 2e. CREATE TABLE IF NOT EXISTS — Ignore mode

In [ ]:
%%sql
DROP TABLE IF EXISTS spanner.demo_catalog_ign

In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS spanner.demo_catalog_ign (
    id   BIGINT NOT NULL,
    note STRING
) USING `cloud-spanner`
TBLPROPERTIES('primaryKeys' = 'id')

In [ ]:
%%sql
INSERT INTO spanner.demo_catalog_ign VALUES (1, 'first')

In [ ]:
%%sql
-- Second CREATE TABLE IF NOT EXISTS is a no-op
CREATE TABLE IF NOT EXISTS spanner.demo_catalog_ign (
    id   BIGINT NOT NULL,
    note STRING
) USING `cloud-spanner`
TBLPROPERTIES('primaryKeys' = 'id')

In [ ]:
%%sql
SELECT * FROM spanner.demo_catalog_ign ORDER BY id

---
# Part 3 — Struct-to-JSON Mapping

Spanner `JSON` columns map to Spark `StringType`. To write structured data:
1. Build a DataFrame with a struct column
2. Convert the struct to a JSON string with `to_json()`
3. Write the string to the Spanner JSON column

To read it back, use `from_json()` to parse the JSON string into a struct.

## 3a. Create a table with a JSON column

In [ ]:
%%sql
DROP TABLE IF EXISTS spanner.demo_json

In [ ]:
%%sql
CREATE TABLE spanner.demo_json (
    id       BIGINT NOT NULL,
    name     STRING,
    metadata STRING
) USING `cloud-spanner`
TBLPROPERTIES('primaryKeys' = 'id')

## 3b. Build a DataFrame with a struct column and convert to JSON

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, ArrayType

# Define a schema with a nested struct column
nested_schema = StructType([
    StructField("id",   LongType(), False),
    StructField("name", StringType(), True),
    StructField("metadata", StructType([
        StructField("role",       StringType(), True),
        StructField("level",      IntegerType(), True),
        StructField("skills",     ArrayType(StringType()), True),
    ]), True),
])

data = [
    (1, "Alice",   ("engineer",  5, ["python", "spark", "sql"])),
    (2, "Bob",     ("analyst",   3, ["sql", "tableau"])),
    (3, "Charlie", ("manager",   7, ["leadership", "strategy"])),
]
struct_df = spark.createDataFrame(data, nested_schema)

print("Original DataFrame with struct column:")
struct_df.printSchema()
struct_df.show(truncate=False)

In [ ]:
# Convert the struct column to a JSON string for writing to Spanner
write_df = struct_df.withColumn("metadata", F.to_json("metadata"))

print("DataFrame after to_json() — ready to write to Spanner:")
write_df.printSchema()
write_df.show(truncate=False)

## 3c. Write to Spanner

In [ ]:
(write_df.write.format("cloud-spanner")
    .options(**SPANNER_OPTS)
    .option("table", "demo_json")
    .mode("append")
    .save())

print("✓ Wrote 3 rows with JSON metadata to Spanner.")

## 3d. Read back — raw JSON strings

In [ ]:
%%sql
SELECT * FROM spanner.demo_json ORDER BY id

## 3e. Parse JSON back into structs with `from_json()`

In [ ]:
# Read via DataFrame API
raw_df = (spark.read.format("cloud-spanner")
    .options(**SPANNER_OPTS)
    .option("table", "demo_json")
    .load()
    .orderBy("id"))

# Define the expected struct schema for parsing
json_schema = StructType([
    StructField("role",       StringType(), True),
    StructField("level",      IntegerType(), True),
    StructField("skills",     ArrayType(StringType()), True),
])

# Parse the JSON string back into a struct
parsed_df = raw_df.withColumn("metadata", F.from_json("metadata", json_schema))

print("Parsed DataFrame — struct column restored:")
parsed_df.printSchema()
parsed_df.show(truncate=False)

In [ ]:
# Access nested fields directly
parsed_df.select(
    "id",
    "name",
    F.col("metadata.role").alias("role"),
    F.col("metadata.level").alias("level"),
    F.explode("metadata.skills").alias("skill"),
).show(truncate=False)

---
## Cleanup

In [ ]:
%%sql
DROP TABLE IF EXISTS spanner.demo_df_write

In [ ]:
%%sql
DROP TABLE IF EXISTS spanner.demo_catalog

In [ ]:
%%sql
DROP TABLE IF EXISTS spanner.demo_overwrite

In [ ]:
%%sql
DROP TABLE IF EXISTS spanner.demo_catalog_eie

In [ ]:
%%sql
DROP TABLE IF EXISTS spanner.demo_catalog_ign

In [ ]:
%%sql
DROP TABLE IF EXISTS spanner.demo_json